# Pricing A Callable Bond 

- ```Discount_Curve```: discount_curve_2025-02-13.xlsx 
- ```Main analysis```: callable_bonds_2025-02-13.xlsx 
- ```Extension problem```: callable_bonds_2025-02-18.xlsx

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1.1.
Consider a bond with:
- ```T = 3```
- Face value of ``` N=100```
- Coupons at ```annual``` frequency
- annualized coupon rate of ```cpn=6%```

In [4]:
bond_df = pd.read_excel('../Data/callable_bonds_2025-02-13.xlsx')
discount_df = pd.read_excel('../Data/discount_curve_2025-02-13.xlsx')

In [5]:
discount_df.head(10)

,ttm,maturity date,spot rate,discount
0,0.5,2025-08-13,0.043743,0.978597
1,1.0,2026-02-13,0.042890,0.958451
2,1.5,2026-08-13,0.042238,0.939228
3,2.0,2027-02-13,0.041843,0.920515
4,2.5,2027-08-13,0.041632,0.902117
5,3.0,2028-02-13,0.041492,0.884084
6,3.5,2028-08-13,0.041412,0.866352
7,4.0,2029-02-13,0.041354,0.848973
8,4.5,2029-08-13,0.041338,0.831832
9,5.0,2030-02-13,0.041327,0.815030


In [31]:
def price_bond(discounts: pd.DataFrame, cpnrate: float, ttm: float, cpnfreq: int = 2, face: float = 100) -> float:
    """
    Prices a typical coupon bond using discount factors.
    
    Parameters:
        discounts (pd.DataFrame): A dataframe with an index representing time-to-maturity (in years)
                                  at intervals (e.g., 0.5, 1.0, ..., 30), and with columns 'spot rate' and 'discount'.
        cpnrate (float): Annual coupon rate. If the value is greater than 1, it is assumed to be a percentage
                         and will be divided by 100.
        ttm (float): Time-to-maturity (in years) of the bond.
        cpnfreq (int, optional): Number of coupon payments per year (default=2 for semiannual coupons).
        face (float, optional): Face (par) value of the bond (default=100).
    
    Returns:
        float: The calculated bond price.
    """
    
    # Adjust coupon rate if provided as a percentage
    if cpnrate > 1:
        cpnrate = cpnrate / 100.0

    coupon_payment = face * cpnrate / cpnfreq

    # If time-to-maturity is less than one coupon period, set payment_dates to just [ttm]
    if ttm < 1/cpnfreq:
        payment_dates = np.array([ttm])
    else:
        # Generate regular coupon payment dates
        payment_dates = np.arange(1/cpnfreq, ttm + 1e-8, 1/cpnfreq)
        # If the last payment date is not exactly ttm, add the maturity as an irregular final period.
        if not np.isclose(payment_dates[-1], ttm):
            payment_dates = np.append(payment_dates, ttm)
    
    price = 0.0
    for t in payment_dates:
        # Determine cash flow:
        # For the final payment at maturity, add the face value.
        # Note: Depending on the bond's terms, you might want to prorate the coupon for an irregular period.
        if np.isclose(t, ttm):
            cash_flow = coupon_payment + face
        else:
            cash_flow = coupon_payment
        
        # Retrieve the discount factor: if an exact match is not found, interpolate.
        if t in discounts.index:
            discount_factor = discounts.loc[t, 'discount']
        else:
            discount_factor = np.interp(t, discounts.index.values, discounts['discount'].values)
        
        price += cash_flow * discount_factor
    
    return price


bond_price = price_bond(discount_df.set_index('ttm'), cpnrate=0.06, ttm=3.0, cpnfreq=1)
print(f"The price of the bond is: {bond_price:.2f}")

The price of the bond is: 104.99


- The price of the bond is 104.987

### 1.2
- European Style
- Expiration of ```topt=1.5```
- (clean) ```strike = 100```
- vol of ```2.68%```
- forward price of ```103.31```

In [10]:
bond_df.head(50)

,info,FHLMC 0.97 01/28/28,FHLMC 1 1/4 01/29/30,FHLMC 4.41 01/28/30
0,CUSIP,3134GW5F9,3134GWGK6,3134HA4V2
1,Issuer,FREDDIE MAC,FREDDIE MAC,FREDDIE MAC
2,Maturity Type,CALLABLE,CALLABLE,CALLABLE
3,Issuer Industry,GOVT AGENCY,GOVT AGENCY,GOVT AGENCY
4,Amount Issued,30000000,25000000,10000000
5,Cpn Rate,0.0097,0.0125,0.0441
6,Cpn Freq,2,2,2
7,Date Quoted,2025-02-13 00:00:00,2025-02-13 00:00:00,2025-02-13 00:00:00
8,Date Issued,2020-10-28 00:00:00,2020-07-29 00:00:00,2025-01-28 00:00:00
9,Date Matures,2028-01-28 00:00:00,2030-01-29 00:00:00,2030-01-28 00:00:00


In [32]:
from scipy.stats import norm

Z = 0.939228  # discount factor at ttm=1.5

d1 = (np.log(103.31/100) + 0.0268**2 * 1.5 / 2) / (0.0268 * np.sqrt(1.5))
d2 = d1 - 0.0268 * np.sqrt(1.5)

call_price = Z * (103.31 * norm.cdf(d1) - 100 * norm.cdf(d2))

print(f"The price of the call option is : {call_price:.2f}")

The price of the call option is : 3.37


### 1.3
callable_bond_price = bond_price - call_price

In [33]:
callable_bond_price = bond_price - call_price
print(f"The price of the callable bond is: {callable_bond_price:.2f}")

The price of the callable bond is: 101.61
